In [42]:
!pip install -q lightgbm

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

In [2]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
OPTION_COLS = ['A', 'B', 'C', 'D', 'E']
print(train.shape, test.shape)

(2000, 8) (500, 7)


In [4]:
def apk(actual, predicted, k=3):
    score, hits = 0.0, 0
    for i, p in enumerate(predicted[:k]):
        if p == actual:
            hits += 1
            score += hits / (i + 1.0)
    return score

def mapk(actuals, preds):
    return float(np.mean([apk(a, p) for a, p in zip(actuals, preds)]))

In [5]:
rows = []
for _, r in train.iterrows():
    for c in OPTION_COLS:
        rows.append({
            'text':   str(r['prompt']) + ' ' + str(r[c]),
            'qid':    r['id'],
            'option': c,
            'label':  1 if r['answer'] == c else 0
        })
pw_train = pd.DataFrame(rows)

rows = []
for _, r in test.iterrows():
    for c in OPTION_COLS:
        rows.append({
            'text':   str(r['prompt']) + ' ' + str(r[c]),
            'qid':    r['id'],
            'option': c
        })
pw_test = pd.DataFrame(rows)

print(pw_train.shape, pw_test.shape)

(10000, 4) (2500, 3)


In [6]:
tfidf  = TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True)
X_tr   = tfidf.fit_transform(pw_train['text'])
X_tst  = tfidf.transform(pw_test['text'])
y_tr   = pw_train['label'].values
groups = pw_train['qid'].values

In [7]:
def rank_preds(pw, scores, orig_df):
    pw = pw.copy()
    pw['score'] = scores
    return [
        pw[pw['qid'] == qid].sort_values('score', ascending=False)['option'].tolist()[:3]
        for qid in orig_df['id']
    ]

gkf      = GroupKFold(n_splits=5)
oof      = np.zeros(len(pw_train))
tst_lgbm = np.zeros(len(pw_test))

for fold, (ti, vi) in enumerate(gkf.split(X_tr, y_tr, groups)):
    model = LGBMClassifier(
        n_estimators=200,
        learning_rate=0.1,
        num_leaves=31,
        random_state=42,
        verbosity=-1
    )
    model.fit(X_tr[ti], y_tr[ti])
    oof[vi]   = model.predict_proba(X_tr[vi])[:, 1]
    tst_lgbm += model.predict_proba(X_tst)[:, 1] / 5

    val_qids   = pw_train.iloc[vi]['qid'].unique()
    val_train  = train[train['id'].isin(val_qids)]
    val_pw     = pw_train[pw_train['qid'].isin(val_qids)]
    val_scores = oof[pw_train['qid'].isin(val_qids).values]
    fold_map3  = mapk(val_train['answer'].tolist(), rank_preds(val_pw, val_scores, val_train))
    print(f"Fold {fold+1}  MAP@3={fold_map3:.4f}")

oof_map3 = mapk(train['answer'].tolist(), rank_preds(pw_train, oof, train))
print(f"\nOOF MAP@3: {oof_map3:.4f}")

Fold 1  MAP@3=0.9729
Fold 2  MAP@3=0.9633
Fold 3  MAP@3=0.9496
Fold 4  MAP@3=0.9563
Fold 5  MAP@3=0.9683

OOF MAP@3: 0.9621


In [8]:
test_preds = rank_preds(pw_test, tst_lgbm, test)

submission = pd.DataFrame({
    'ID':         test['id'],
    'Prediction': [' '.join(p) for p in test_preds]
})

submission.to_csv('submission.csv', index=False)
print(submission.head(10).to_string(index=False))

 ID Prediction
  1      A E B
  2      B E D
  3      B E D
  4      E C D
  5      C D B
  6      D A B
  7      E D C
  8      B E C
  9      C D E
 10      B E A
